## Import Dataset

In [1]:
import pandas as pd
df = pd.read_csv("../../raw_data/recipes_ingredients.csv")

## Checking stuff

In [2]:
df.shape

(500471, 9)

In [3]:
df.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## Preprocessing

In [4]:
data = df.dropna()  #enlève les NaN

In [5]:
data = data.drop(columns=["description","id"])  # on enlève les colonnes innutiles
data.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,Cherry Streusel Cobbler,"[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,Reuben and Swiss Casserole Bake,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,Yam-Pecan Recipe,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,Tropical Orange Layer Cake,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## Fonction pour transformer les strings en liste.
## on applique sur les colonnes concernés (ingredients, ingredients_raw, steps, tags)

In [6]:
import ast

errors = []

def safe_literal_eval(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError) as e:
        errors.append((value, str(e)))
        return value

data["ingredients"] = data["ingredients"].apply(safe_literal_eval)
data["ingredients_raw"] = data["ingredients_raw"].apply(safe_literal_eval)
data["steps"] = data["steps"].apply(safe_literal_eval)
data["tags"] = data["tags"].apply(safe_literal_eval)


In [7]:
type(data.iloc[0].steps) #vérification

list

In [8]:
data.shape #shape avant

(497563, 7)

## On enlève les lignes sans ingrédients, ingredients_raw, steps et tags

In [9]:
data = data[data["ingredients"].apply(len) > 0]
data = data[data["ingredients_raw"].apply(len) > 0]
data = data[data["steps"].apply(len) > 0]
data = data[data["tags"].apply(len) > 0]

In [10]:
data.shape

(489757, 7)

## On teste avec une liste d'ingrédient fictive

In [11]:
user_ingredients = ["tomato", "chicken", "onion", "garlic", "apple", "sugar", "cheese", "bread"]
top_recipes = data.copy() # New Dataset pour rank en fonction de la liste

# Fonction pour compter le nombre de matchs entre la liste et le dataset

In [12]:
def count_matches(recipe_ingredients, user_ingredients):
    matches = 0

    for user_ing in user_ingredients:
        for recipe_ing in recipe_ingredients:
            if user_ing.lower() in recipe_ing.lower():
                matches += 1
                break

    return matches

# Ajoute au nouveau dataset , une colonne match count pour compter le nombre de matchs

In [13]:
top_recipes["match_count"] = top_recipes["ingredients"].apply(lambda x: count_matches(x, user_ingredients))

In [14]:
top_recipes.sort_values("match_count", ascending=False, inplace=True)

In [15]:
top_recipes.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags,match_count
234291,BBQ Chicken Packed Pita,[boneless skinless chicken thighs breasts thig...,[1 lb boneless skinless chicken thighs (o...,[Prepare a hot grill outside or preheat a gril...,1.0,1 (4047 g),"[weeknight, 60-minutes-or-less, time-to-make, ...",8
109082,Fajita Pita,"[chicken breast, soy sauce, lime juice, canola...","[1 lb chicken breast, skinless, boneless ...","[""Whisk together the marinade ingredients in a...",3.0,1 (296 g),"[time-to-make, main-ingredient, cuisine, prepa...",7
140096,Ultimate Chicken Parmigiana,"[virgin olive oil, virgin olive oil, medium on...","[1/4 cup extra virgin olive oil, plus , 3 ...","[Preheat the oven to 350 degrees F., Coat a sa...",4.0,1 (864 g),"[time-to-make, course, main-ingredient, cuisin...",7
185243,Chicken Parmigiana - Tyler Florence,"[virgin olive oil, medium onion chopped, salt,...","[1/2 cup extra virgin olive oil, divided ,...","[In a large skillet, heat 1/4 c olive oil over...",4.0,1 (809 g),"[celebrity, 60-minutes-or-less, time-to-make, ...",7
391728,Chicken Parmigiana,"[olive oil, brown onion, garlic clove minced, ...","[1 teaspoon olive oil, 1 brown onio...","[Heat oil in a small saucepan on medium, and a...",4.0,1 (539 g),"[time-to-make, course, main-ingredient, prepar...",7


# vérification des ingrédients à la main

In [16]:
liste = top_recipes.iloc[1].ingredients
liste

['chicken breast',
 'soy sauce',
 'lime juice',
 'canola oil',
 'garlic minced',
 'brown sugar',
 'cumin',
 'chili powder',
 'cheddar cheese shredded',
 'onions',
 'salt pepper',
 'tomatoes beefsteak',
 'iceberg lettuce',
 'pita bread']

In [17]:
top_recipes.iloc[0].steps

['Prepare a hot grill outside or preheat a grill pan to high heat and preheat oven to 375 degrees F.',
 'Toss raw chicken in a bowl with rub, a few pieces at a time until all chicken is well coated. Grill chicken on well oiled grill or grill pan for about 2 minutes on each side, until nice grill marks are formed. If chicken pieces are very thick, finish cooking for 5 to 10 minutes more in the preheated oven. Allow chicken to cool a few minutes and then pull apart into nice shreds. Toss chicken with BBQ sauce so that it is well coated but not overly wet. Keep warm in a warm saute pan until ready to use.',
 'Heat olive oil in large saute pan over medium-high heat. Stir in garlic with wooden spoon. After 30 seconds add spinach and turn off heat. Toss spinach in hot oil, when wilted completely season with salt. Set aside.',
 'Spread butter evenly over 1 side of each pita. In a large saute pan, grill each flatbread over medium-high heat, buttered side down (work in batches if necessary) top

# Diviser serving_size en 2 colonnes distinctes

In [18]:
data.head(1)

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,1 (347 g),"[60-minutes-or-less, time-to-make, course, mai..."


In [19]:
data[["persons", "portion_size"]] = data["serving_size"].str.extract(
    r"(\d+)\s*\(([^)]+)\)")
data.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags,persons,portion_size
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,1 (347 g),"[60-minutes-or-less, time-to-make, course, mai...",1,347 g
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,1 (207 g),"[60-minutes-or-less, time-to-make, course, mai...",1,207 g
2,Yam-Pecan Recipe,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,1 (198 g),"[time-to-make, course, main-ingredient, cuisin...",1,198 g
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,1 (191 g),"[60-minutes-or-less, time-to-make, course, pre...",1,191 g
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,1 (26 g),"[15-minutes-or-less, time-to-make, course, mai...",1,26 g


In [20]:
data = data.drop(columns="serving_size", axis=1)
data.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,"[60-minutes-or-less, time-to-make, course, mai...",1,347 g
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,"[60-minutes-or-less, time-to-make, course, mai...",1,207 g
2,Yam-Pecan Recipe,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,"[time-to-make, course, main-ingredient, cuisin...",1,198 g
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,"[60-minutes-or-less, time-to-make, course, pre...",1,191 g
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,"[15-minutes-or-less, time-to-make, course, mai...",1,26 g


In [21]:
data = data[data["servings"] <= 50]
data.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,"[60-minutes-or-less, time-to-make, course, mai...",1,347 g
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,"[60-minutes-or-less, time-to-make, course, mai...",1,207 g
2,Yam-Pecan Recipe,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,"[time-to-make, course, main-ingredient, cuisin...",1,198 g
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,"[60-minutes-or-less, time-to-make, course, pre...",1,191 g
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,"[15-minutes-or-less, time-to-make, course, mai...",1,26 g
